In [1]:
#importing the necessary libraries and modules

import torch
import torchvision
import torchvision.models as models
import torchvision.transforms as transforms
import copy
import matplotlib.pyplot as plt
import numpy
from PIL import Image
import torch.nn as nn

device = 'cuda'
print(device)
img_size = (720, 1280)

cuda


In [2]:
#initializing mean and std of ImageNet and constructing the pipeline to convert, resize, and normalize the 
# PIL image. Also, defining a deprocess function to convert a given tensor into image

mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

transform = transforms.Compose([transforms.Resize(img_size), transforms.ToTensor(),
                                transforms.Normalize(mean, std)])

def load_image(path):
    x = Image.open(path)
    x = transform(x)
    y = x.unsqueeze(0)
    y = y.to(device)
    return y

def deprocess(tensor):
    tensor = tensor.squeeze(0)
    std1 = torch.tensor(std).reshape(3, 1, 1).to(device)
    mean1 = torch.tensor(mean).reshape(3, 1, 1).to(device)
    tensor = tensor * std1 + mean1
    tensor = tensor.clamp(0, 1)
    img = transforms.ToPILImage()(tensor)
    return img
    

In [3]:
#importing the model needed for this project - VGG19

vgg = models.vgg19(pretrained = True).features.to(device)

for p in vgg.parameters():
    p.requires_grad = False

vgg = vgg.eval()

/home/divya/.local/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/divya/.local/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [4]:
vgg

#Architecture of VGG. Indices for the required layers:
#                     conv1_1 : 0
#                     conv2_1 : 5
#                     conv3_1 : 10
#                     conv4_1 : 19
#                     conv4_2 : 21
#                     conv5_1 : 28

Sequential(
  (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU(inplace=True)
  (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (3): ReLU(inplace=True)
  (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (6): ReLU(inplace=True)
  (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (8): ReLU(inplace=True)
  (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (11): ReLU(inplace=True)
  (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (13): ReLU(inplace=True)
  (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (15): ReLU(inplace=True)
  (16): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (17): ReLU(inplace=True)
  (18): MaxPoo

In [5]:
def feature_maps(tensor, vgg):
    i = 0
    j = 0
    features = {}
    for layer in vgg:
        tensor = layer(tensor)
        if i == 0 or i == 5 or i == 10 or i == 19:
            j+=1
            features[f"conv{j}_1"] = tensor
        elif i == 21:
            features["conv4_2"] = tensor
        elif i == 28:
            features["conv5_1"] = tensor
            break
        else:
            pass
        i+=1
    
    return features

In [6]:
def gram_matrix(tensor):
    tensor = tensor.squeeze(0)
    N, H, W = tensor.shape
    t1 = tensor.reshape(N, H*W)
    t2 = t1 @ t1.T
    return t2

def content_loss_fn(F_x, F_c):
    content_loss = 0.5 * nn.MSELoss(reduction = 'sum')(F_x, F_c)
    return content_loss

def style_loss_fn(features_x, features_s):
    style_loss = 0
    for layer in features_x:
        if layer == 'conv4_2':
            continue
        gram_x = gram_matrix(features_x[layer])
        gram_s = gram_matrix(features_s[layer])
        _, N, H, W = features_x[layer].shape
        M = H*W
        style_loss += 0.20 * 1/(4*(N**2)*(M**2)) * nn.MSELoss(reduction = 'sum')(gram_x, gram_s)
    
    return style_loss  

def total_loss_fn(content, style, alpha, beta):
    return alpha*content + beta*style


In [7]:
content = load_image('content/tokyo.jpg')
style = load_image('style/starry_night.jpg')

content_features = feature_maps(content, vgg)
style_features = feature_maps(style, vgg)
x = content.clone().detach()
x.requires_grad = True

In [ ]:
optimizer = torch.optim.Adam([x], lr = 0.01)
alpha = 0.01
beta = 1e7
steps = 10000

for i in range(steps):
    optimizer.zero_grad()
    F_c = content_features['conv4_2']
    feature_x = feature_maps(x, vgg)
    content_loss = content_loss_fn(feature_x['conv4_2'], F_c)
    style_loss = style_loss_fn(feature_x, style_features)
    total_loss = total_loss_fn(content_loss, style_loss, alpha, beta)
    total_loss.backward()
    optimizer.step()
    x.data.clamp_(0,1)
    if (i+1) % 50 == 0:
        print(f"{i+1} : {total_loss}")
    
    
    

50 : 4975534.5
100 : 3411365.0
150 : 2605270.0
200 : 2128037.5
250 : 1779363.0
300 : 1543295.75
350 : 1372532.625
400 : 1237186.0
450 : 1133058.5
500 : 1051963.125
550 : 978886.375
600 : 920696.875
650 : 887897.375
700 : 841941.375
750 : 796154.75
800 : 762604.1875
850 : 734157.625
900 : 709181.5
950 : 1609117.25
1000 : 686210.25
1050 : 654714.75
1100 : 634550.8125
1150 : 618035.25
1200 : 603554.625
1250 : 590480.125
1300 : 578527.9375
1350 : 612807.3125
1400 : 568518.625
1450 : 551831.125
1500 : 540261.4375
1550 : 530585.875
1600 : 521963.6875
1650 : 514063.5
1700 : 543350.1875
1750 : 511701.53125
1800 : 499068.625
1850 : 490259.5
1900 : 482999.78125
1950 : 476603.25


In [ ]:
plt.imshow(deprocess(x))
plt.axis('off')
plt.show()

In [ ]:
output = deprocess(x)
output.save("hero.png")